# Notebook 3: Protein Thermostability Prediction

Runs Biopython ProtParam on amino acid sequences stored in PlasticDB.
Computes instability index, GRAVY score, isoelectric point, and aromaticity
for each sequence that has a thermophilic label, then compares thermophilic vs
mesophilic groups statistically.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "plastic-biodegradation-analysis"))
from src.data_loader import load_plasticdb
from Bio.SeqUtils.ProtParam import ProteinAnalysis
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats

df = load_plasticdb()
seq_df = df[df["has_sequence"]].copy()
seq_df["seq_clean"] = seq_df["sequence"].str.upper().str.replace(r"[^ACDEFGHIKLMNPQRSTVWY]", "", regex=True)
seq_df = seq_df[seq_df["seq_clean"].str.len() >= 50].copy()
print(f"Sequences available for ProtParam: {len(seq_df):,}")


In [ ]:
records = []
for _, row in seq_df.iterrows():
    try:
        pa = ProteinAnalysis(row["seq_clean"])
        records.append({
            "organism":        row["organism"],
            "plastic":         row["plastic"],
            "thermophilic":    row["thermophilic"],
            "seq_length":      len(row["seq_clean"]),
            "mol_weight_kda":  pa.molecular_weight() / 1000,
            "isoelectric_pt":  pa.isoelectric_point(),
            "instability_idx": pa.instability_index(),
            "gravy":           pa.gravy(),
            "aromaticity":     pa.aromaticity(),
            "predicted_stable": pa.instability_index() < 40,
        })
    except Exception:
        pass

prop_df = pd.DataFrame(records)
print(f"Successfully computed: {len(prop_df):,} sequences")
print(prop_df.describe().round(3).to_string())


## Statistical comparison: thermophilic vs mesophilic

In [ ]:
labelled = prop_df[prop_df["thermophilic"].notna()].copy()
labelled["label"] = labelled["thermophilic"].map({True: "Thermophilic", False: "Mesophilic"})

for metric in ["instability_idx", "gravy", "mol_weight_kda", "isoelectric_pt"]:
    thermo = labelled[labelled["label"]=="Thermophilic"][metric].dropna()
    meso   = labelled[labelled["label"]=="Mesophilic"][metric].dropna()
    if len(thermo) == 0 or len(meso) == 0:
        print(f"{metric}: insufficient data")
        continue
    stat, p = stats.mannwhitneyu(thermo, meso, alternative="two-sided")
    print(f"{metric:20s}  thermo mean={thermo.mean():.3f}  meso mean={meso.mean():.3f}  p={p:.4f}")


## Stability rate

In [ ]:
stab = labelled.groupby("label")["predicted_stable"].mean() * 100
print("Stability rate (instability index < 40):")
print(stab.round(1).to_string())


## Visualisation

In [ ]:
COLORS = {"Thermophilic": "#F44336", "Mesophilic": "#2196F3"}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("ProtParam Thermostability Analysis of PlasticDB Sequences", fontsize=13, fontweight="bold")

ax = axes[0,0]
for label, color in COLORS.items():
    sub = labelled[labelled["label"]==label]["instability_idx"].dropna()
    ax.hist(sub, bins=30, alpha=0.65, color=color, label=f"{label} (n={len(sub)})", edgecolor="white")
ax.axvline(40, color="black", linestyle="--", label="Stability threshold (40)")
ax.set_xlabel("Instability index"); ax.set_ylabel("Count")
ax.set_title("A. Instability Index", fontweight="bold")
ax.legend(fontsize=8); ax.spines[["top","right"]].set_visible(False)

ax = axes[0,1]
for label, color in COLORS.items():
    sub = labelled[labelled["label"]==label]["gravy"].dropna()
    ax.hist(sub, bins=30, alpha=0.65, color=color, label=label, edgecolor="white")
ax.axvline(0, color="black", linestyle="--")
ax.set_xlabel("GRAVY score"); ax.set_ylabel("Count")
ax.set_title("B. GRAVY Score", fontweight="bold")
ax.legend(fontsize=8); ax.spines[["top","right"]].set_visible(False)

ax = axes[1,0]
stab = labelled.groupby("label")["predicted_stable"].mean()*100
ax.bar(stab.index, stab.values, color=[COLORS[l] for l in stab.index], edgecolor="black")
for l, v in stab.items():
    ax.text(l, v+0.5, f"{v:.1f}%", ha="center", fontsize=11, fontweight="bold")
ax.set_ylabel("% predicted stable")
ax.set_title("C. Predicted Stability Rate", fontweight="bold"); ax.set_ylim(0, 100)
ax.spines[["top","right"]].set_visible(False)

ax = axes[1,1]
for label, color in COLORS.items():
    sub = labelled[labelled["label"]==label]
    ax.scatter(sub["gravy"], sub["instability_idx"], alpha=0.35, color=color, s=12, label=label)
ax.axhline(40, color="black", linestyle="--", label="Stability threshold")
ax.set_xlabel("GRAVY score"); ax.set_ylabel("Instability index")
ax.set_title("D. GRAVY vs Instability", fontweight="bold")
ax.legend(fontsize=8); ax.spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.savefig("../outputs/figures/03_thermostability_prediction.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved.")
